In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
for df in (df_metric, df_trace, df_log, df_err):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregates
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name']).size().reset_index(name='count')
metric_summary = metric_grp.sort_values('count', ascending=False).head(50)
metric_total_rows = df_metric.shape[0]
metric_unique_pairs = metric_grp.shape[0]

# 2) trace.csv aggregates
trace_grp = df_trace.groupby(['cmdb_id', 'trace_name']).size().reset_index(name='count')
trace_summary = trace_grp.sort_values('count', ascending=False).head(50)
trace_total_rows = df_trace.shape[0]
trace_unique_pairs = trace_grp.shape[0]

# 3) log.csv aggregates
log_grp = df_log.groupby(['cmdb_id', 'log_name']).size().reset_index(name='count')
log_summary = log_grp.sort_values('count', ascending=False).head(50)
log_total_rows = df_log.shape[0]
log_unique_pairs = log_grp.shape[0]

# 4) error_logs.csv summary
err_total_rows = df_err.shape[0]
err_counts_per_cmdb = df_err.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
err_top10 = df_err.sort_values('ts', ascending=False)[['ts', 'cmdb_id', 'message']].head(10)

# Display compact results (variables separated by commas for IPython)
metric_summary, metric_total_rows, metric_unique_pairs, \
trace_summary, trace_total_rows, trace_unique_pairs, \
log_summary, log_total_rows, log_unique_pairs, \
err_total_rows, err_counts_per_cmdb, err_top10

```
Out[1]:
```
# Summarize the previously computed aggregates into a concise plain-English summary string.
# Reuse existing variables from prior execution: metric_total_rows, metric_unique_pairs, metric_summary,
# trace_total_rows, trace_unique_pairs, trace_summary, log_total_rows, log_unique_pairs, log_summary,
# err_total_rows, err_counts_per_cmdb.

def top_pairs_to_lines(df, n=5, cols=None):
    if df is None or df.shape[0]==0:
        return ["(none)"]
    cols = cols or df.columns.tolist()
    lines = []
    for i, row in df.head(n).iterrows():
        # format: rank) cmdb_id | name | count
        name = row[cols[1]] if len(cols)>1 else row[cols[0]]
        lines.append(f"{i+1}) {row[cols[0]]} | {name} | count={row['count']}")
    return lines

metric_lines = top_pairs_to_lines(metric_summary.reset_index(drop=True), n=5, cols=['cmdb_id','kpi_name'])
trace_lines = top_pairs_to_lines(trace_summary.reset_index(drop=True), n=5, cols=['cmdb_id','trace_name'])
log_lines = top_pairs_to_lines(log_summary.reset_index(drop=True), n=8, cols=['cmdb_id','log_name'])  # show top 8 due to many ties

if err_total_rows == 0:
    err_summary_lines = ["error_logs.csv contains 0 rows (no error logs)."]
else:
    # compact list counts per cmdb_id
    err_counts_lines = [f"{r['cmdb_id']}: {r['count']}" for _, r in err_counts_per_cmdb.iterrows()]
    err_summary_lines = [f"error_logs total rows = {err_total_rows}", "counts per cmdb_id:"] + err_counts_lines[:20]

summary = (
    f"Summary of telemetry files (UTC+8):\n\n"
    f"1) metric.csv\n"
    f"- Total rows: {metric_total_rows}\n"
    f"- Unique (cmdb_id, kpi_name) pairs: {metric_unique_pairs}\n"
    f"- Top pairs by row count (showing up to 5 of the returned top 50):\n  " + "\n  ".join(metric_lines) + "\n\n"
    f"2) trace.csv\n"
    f"- Total rows: {trace_total_rows}\n"
    f"- Unique (cmdb_id, trace_name) pairs: {trace_unique_pairs}\n"
    f"- Top pairs by row count (showing up to 5 of the returned top 50):\n  " + "\n  ".join(trace_lines) + "\n\n"
    f"3) log.csv\n"
    f"- Total rows: {log_total_rows}\n"
    f"- Unique (cmdb_id, log_name) pairs: {log_unique_pairs}\n"
    f"- Top pairs by row count (showing up to 8):\n  " + "\n  ".join(log_lines) + "\n\n"
    f"4) error_logs.csv\n"
    + "\n  ".join(err_summary_lines)
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count
6       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
2       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
15      IG01                             OSLinux-CPU_CPU_CPUWio     31
..       ...                                                ...    ...
820  Redis01                        OSLinux-CPU_CPU_CPUUserTime     31
821  Redis01                             OSLinux-CPU_CPU_CPUWio     31
895  Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
884  Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31
887  Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31

[50 rows x 3 columns], 27487, 1705,    cmdb_id                         trace_name  count
5     IG01               trace.self.row_count     30
4     IG01            trace.self.duration_p95     30
3     IG01           trace.self.duration_mean     30
6     IG01    trace.to_Tomcat01.duration_mean     30
9     IG01    trace.to_Tomcat02.duration_mean     30
..     ...                                ...    ...
45    MG01  trace.from_Tomcat04.duration_mean     30
46    MG01   trace.from_Tomcat04.duration_p95     30
55    MG01   trace.from_dockerB1.duration_p95     30
56    MG01      trace.from_dockerB1.row_count     30
57    MG01  trace.from_dockerB2.duration_mean     30

[50 rows x 3 columns], 6879, 255,      cmdb_id          log_name  count
0   Tomcat01   log.error_count     31
2   Tomcat01     log.row_count     31
3   Tomcat02   log.error_count     31
5   Tomcat02     log.row_count     31
6   Tomcat03   log.error_count     31
11  Tomcat04     log.row_count     31
9   Tomcat04   log.error_count     31
8   Tomcat03     log.row_count     31
14  apache02   log.error_count     31
13  apache01     log.row_count     31
15  apache02     log.row_count     31
12  apache01   log.error_count     31
4   Tomcat02  log.gc_pause_max     20
1   Tomcat01  log.gc_pause_max     11
7   Tomcat03  log.gc_pause_max     11
10  Tomcat04  log.gc_pause_max     11, 425, 16, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware timestamp column 'ts' in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if df_metric['ts'].dt.tz is None:
        df_metric['ts'] = pd.to_datetime(df_metric['ts']).dt.tz_localize(tz)
    else:
        df_metric['ts'] = df_metric['ts'].dt.tz_convert(tz)

# 1) Compute global thresholds (total_count, global P95, global P90) for each (cmdb_id, kpi_name)
metrics_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(total_count=('count'),
         p95=lambda x: x.quantile(0.95),
         p90=lambda x: x.quantile(0.90))
)

# 2) Filter metric.csv to incident window (UTC+8): 2021-03-12 20:30 to 2021-03-12 21:00 inclusive
start_ts = pd.Timestamp('2021-03-12 20:30:00', tz=tz)
end_ts = pd.Timestamp('2021-03-12 21:00:00', tz=tz)
df_win = df_metric[(df_metric['ts'] >= start_ts) & (df_metric['ts'] <= end_ts)].copy()

# points_in_window per pair
points_in_window = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(points_in_window=('value','count'))
)

# anomaly counts using global p95: merge p95 into windowed points
df_win_with_p95 = df_win.merge(metrics_thresholds[['cmdb_id','kpi_name','p95']],
                               on=['cmdb_id','kpi_name'],
                               how='left')
# mark anomalies where value >= global p95
df_win_with_p95['is_anom_p95'] = (df_win_with_p95['value'] >= df_win_with_p95['p95']).astype(int)

# aggregate anomaly_count_p95 and earliest_anomaly_ts_p95 per pair
anomaly_agg = (
    df_win_with_p95[df_win_with_p95['is_anom_p95'] == 1]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(anomaly_count_p95=('is_anom_p95','sum'),
         earliest_anomaly_ts_p95=('ts','min'))
)

# 3) Build metrics_window by left-joining thresholds with window aggregates (ensure every pair appears)
metrics_window = (
    metrics_thresholds
    .merge(points_in_window, on=['cmdb_id','kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
)

# Fill missing window values with zeros / NaT as required
metrics_window['points_in_window'] = metrics_window['points_in_window'].fillna(0).astype(int)
metrics_window['anomaly_count_p95'] = metrics_window['anomaly_count_p95'].fillna(0).astype(int)
# earliest_anomaly_ts_p95 keep as datetime (NaT for none)

# Prepare compact top-20 table sorted by anomaly_count_p95 descending
top20_pairs = (
    metrics_window
    .sort_values(['anomaly_count_p95', 'points_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','total_count','p95','p90','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']]
    .head(20)
)

# Count of metric pairs that had at least one P95 anomaly in the window
num_pairs_with_anomaly = int((metrics_window['anomaly_count_p95'] > 0).sum())

# For compactness show shapes of stored full tables plus the top20 and the integer count
metrics_thresholds_shape = metrics_thresholds.shape
metrics_window_shape = metrics_window.shape

metrics_thresholds_shape, metrics_window_shape, top20_pairs, num_pairs_with_anomaly

```
Out[2]:
```
# Compose a concise plain-English summary using existing kernel variables
try:
    metrics_thresholds
    metrics_window
    top20_pairs
    num_pairs_with_anomaly
except NameError:
    raise RuntimeError("Required variables (metrics_thresholds, metrics_window, top20_pairs, num_pairs_with_anomaly) are missing in the kernel.")

summary = (
    "Metric file analysis (incident window 2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Total unique (cmdb_id, kpi_name) metric pairs: {metrics_thresholds.shape[0]}\n"
    f"- Pairs with at least one value >= their global P95 during the window: {num_pairs_with_anomaly}\n"
    "- Observations: many pairs have points_in_window=31 (full window) and anomaly_count_p95=31,\n"
    "  with earliest anomalies often at 2021-03-12 20:30:00+08:00, indicating wide-spread values >= global P95 across metrics.\n"
    "- See top20_pairs for the top 20 (cmdb_id, kpi_name) sorted by anomaly_count_p95."
)

# Display the compact summary, the integer count, and the top-20 compact table
summary, num_pairs_with_anomaly, top20_pairs.head(20)

The original code execution output of IPython Kernel is also provided below for reference:

((1705, 5), (1705, 8),      cmdb_id                                           kpi_name  total_count           p95           p90  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax           31  2.040136e+09  2.040136e+09                31                 31 2021-03-12 20:30:00+08:00
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed           31  5.802017e+07  5.802017e+07                31                 31 2021-03-12 20:30:00+08:00
77      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  1.000000e+00  1.000000e+00                31                 31 2021-03-12 20:30:00+08:00
80      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...           31  3.000000e-01  3.000000e-01                31                 31 2021-03-12 20:30:00+08:00
81      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  3.000000e+00  3.000000e+00                31                 31 2021-03-12 20:30:00+08:00
89      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax           31  2.040136e+09  2.040136e+09                31                 31 2021-03-12 20:30:00+08:00
156     IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
167     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  1.000000e+00  1.000000e+00                31                 31 2021-03-12 20:30:00+08:00
171     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  3.000000e+00  3.000000e+00                31                 31 2021-03-12 20:30:00+08:00
179     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax           31  2.040136e+09  2.040136e+09                31                 31 2021-03-12 20:30:00+08:00
182     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed           31  5.069495e+08  5.069495e+08                31                 31 2021-03-12 20:30:00+08:00
271     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax           31  2.040136e+09  2.040136e+09                31                 31 2021-03-12 20:30:00+08:00
274     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed           31  9.471521e+08  9.471521e+08                31                 31 2021-03-12 20:30:00+08:00
573  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
574  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
576  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
580  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           31  7.000000e+00  7.000000e+00                31                 31 2021-03-12 20:30:00+08:00
798  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
799  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00
801  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           31  0.000000e+00  0.000000e+00                31                 31 2021-03-12 20:30:00+08:00, 1705)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse existing kernel variables: metrics_window, metrics_thresholds
try:
    metrics_window
    metrics_thresholds
except NameError as e:
    raise RuntimeError("Required kernel variables metrics_window and metrics_thresholds are missing") from e

# 1) distinct_kpi_count per cmdb_id from metrics_thresholds
distinct_kpi_count = (
    metrics_thresholds
    .groupby('cmdb_id', as_index=False)
    .agg(distinct_kpi_count=('kpi_name','count'))
)

# 2) aggregate anomalies per cmdb_id
agg_basic = (
    metrics_window
    .groupby('cmdb_id', as_index=False)
    .agg(
        num_kpi_pairs_with_anomaly=('anomaly_count_p95', lambda x: (x>0).sum()),
        total_anomaly_points=('anomaly_count_p95', 'sum')
    )
)

# 3) earliest anomaly timestamp per cmdb_id among kpis that had anomaly_count_p95 > 0
earliest_ts = (
    metrics_window[metrics_window['anomaly_count_p95'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly_ts=('earliest_anomaly_ts_p95', 'min'))
)

# 4) combine into full component-level table
component_anomaly_agg = (
    agg_basic
    .merge(distinct_kpi_count, on='cmdb_id', how='left')
    .merge(earliest_ts, on='cmdb_id', how='left')
)

# Ensure types: fill NaN distinct_kpi_count with 0 if any (unlikely) and convert ints
component_anomaly_agg['distinct_kpi_count'] = component_anomaly_agg['distinct_kpi_count'].fillna(0).astype(int)
component_anomaly_agg['num_kpi_pairs_with_anomaly'] = component_anomaly_agg['num_kpi_pairs_with_anomaly'].astype(int)
component_anomaly_agg['total_anomaly_points'] = component_anomaly_agg['total_anomaly_points'].astype(int)

# 5) Top-20 by total_anomaly_points descending
component_anomaly_agg_top20 = component_anomaly_agg.sort_values('total_anomaly_points', ascending=False).head(20)

# 6) Total number of distinct components that had at least one P95 anomaly in the window
num_components_with_anomaly = int((component_anomaly_agg['num_kpi_pairs_with_anomaly'] > 0).sum())

# Return compact results: top-20 table and the integer count
component_anomaly_agg_top20, num_components_with_anomaly

```
Out[3]:
```
```python
# Compose a concise plain-English summary string using existing kernel variables
try:
    component_anomaly_agg_top20
    num_components_with_anomaly
except NameError:
    raise RuntimeError("Required variables component_anomaly_agg_top20 or num_components_with_anomaly are missing in the kernel.")

top5 = component_anomaly_agg_top20.head(5).reset_index(drop=True)
top5_lines = [
    f"{i+1}. {row['cmdb_id']}: total_anomaly_points={row['total_anomaly_points']}, "
    f"num_kpi_pairs_with_anomaly={row['num_kpi_pairs_with_anomaly']}, "
    f"distinct_kpi_count={row['distinct_kpi_count']}"
    for i, row in top5.iterrows()
]

summary = (
    f"Component-level P95 anomaly summary (incident window 2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Number of components with at least one P95 anomaly: {num_components_with_anomaly}\n"
    f"- Top components by total anomaly points (top 5):\n  " + "\n  ".join(top5_lines) + "\n"
    f"- Earliest anomalies for top components occurred at 2021-03-12 20:30:00+08:00 (observed across many metrics).\n"
    f"- Interpretation: MySQL and Redis components show the highest total anomaly points, followed by multiple Tomcat and infra nodes, "
    f"suggesting widespread metric elevations hitting global P95 thresholds across many KPIs during the window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  num_kpi_pairs_with_anomaly  total_anomaly_points  distinct_kpi_count       earliest_anomaly_ts
4         Mysql01                         226                  2377                 226 2021-03-12 20:30:00+08:00
5         Mysql02                         225                  2187                 225 2021-03-12 20:30:00+08:00
6         Redis01                         115                  1329                 115 2021-03-12 20:30:00+08:00
7         Redis02                         109                  1291                 109 2021-03-12 20:30:00+08:00
21       Tomcat03                         109                  1021                 109 2021-03-12 20:30:00+08:00
20       Tomcat02                         110                  1011                 110 2021-03-12 20:30:00+08:00
22       Tomcat04                         111                   975                 111 2021-03-12 20:30:00+08:00
19       Tomcat01                         107                   940                 107 2021-03-12 20:30:00+08:00
0            IG01                          89                   663                  89 2021-03-12 20:30:00+08:00
2            MG01                          92                   653                  92 2021-03-12 20:30:00+08:00
24       apache02                          82                   620                  82 2021-03-12 20:30:00+08:00
1            IG02                          90                   615                  90 2021-03-12 20:30:00+08:00
23       apache01                          82                   604                  82 2021-03-12 20:30:00+08:00
3            MG02                          91                   588                  91 2021-03-12 20:30:00+08:00
12   ServiceTest3                           4                    66                   4 2021-03-12 20:30:00+08:00
10  ServiceTest11                           4                    66                   4 2021-03-12 20:30:00+08:00
9   ServiceTest10                           4                    66                   4 2021-03-12 20:30:00+08:00
8    ServiceTest1                           4                    66                   4 2021-03-12 20:30:00+08:00
11   ServiceTest2                           4                    66                   4 2021-03-12 20:30:00+08:00
14   ServiceTest5                           4                    66                   4 2021-03-12 20:30:00+08:00, 29)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware timestamp column 'ts' in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_trace['ts'].dt.tz is None:
        df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_localize(tz)
    else:
        df_trace['ts'] = df_trace['ts'].dt.tz_convert(tz)

# 1) Compute global thresholds for each (cmdb_id, trace_name)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(
        total_count=('count'),
        p95=lambda x: x.quantile(0.95),
        p90=lambda x: x.quantile(0.90)
    )
)

# 2) Window filter: 2021-03-12 20:30 to 2021-03-12 21:00 UTC+8
start_ts = pd.Timestamp('2021-03-12 20:30:00', tz=tz)
end_ts = pd.Timestamp('2021-03-12 21:00:00', tz=tz)
df_trace_win = df_trace[(df_trace['ts'] >= start_ts) & (df_trace['ts'] <= end_ts)].copy()

# points_in_window per pair
points_in_window = (
    df_trace_win
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(points_in_window=('value','count'))
)

# Merge global p95 into windowed points and mark anomalies
df_trace_win_p95 = df_trace_win.merge(
    trace_thresholds[['cmdb_id','trace_name','p95']],
    on=['cmdb_id','trace_name'],
    how='left'
)
df_trace_win_p95['is_anom_p95'] = (df_trace_win_p95['value'] >= df_trace_win_p95['p95']).astype(int)

# Aggregate anomaly count and earliest anomaly timestamp per pair
anomaly_agg = (
    df_trace_win_p95[df_trace_win_p95['is_anom_p95'] == 1]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        anomaly_count_p95=('is_anom_p95','sum'),
        earliest_anomaly_ts_p95=('ts','min')
    )
)

# 2) Build trace_window: left-join thresholds with window aggregates so every pair appears
trace_window = (
    trace_thresholds
    .merge(points_in_window, on=['cmdb_id','trace_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id','trace_name'], how='left')
)

# fill missing window values appropriately
trace_window['points_in_window'] = trace_window['points_in_window'].fillna(0).astype(int)
trace_window['anomaly_count_p95'] = trace_window['anomaly_count_p95'].fillna(0).astype(int)
# earliest_anomaly_ts_p95 remains NaT where no anomaly

# 3a) trace_pair_top20: top 20 pairs sorted by anomaly_count_p95 desc
trace_pair_top20 = (
    trace_window
    .sort_values(['anomaly_count_p95','points_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','trace_name','total_count','p95','p90','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']]
    .head(20)
)

# 3b) component-level aggregation
distinct_trace_pair_count = (
    trace_thresholds
    .groupby('cmdb_id', as_index=False)
    .agg(distinct_trace_pair_count=('trace_name','count'))
)

agg_basic = (
    trace_window
    .groupby('cmdb_id', as_index=False)
    .agg(
        num_trace_pairs_with_anomaly=('anomaly_count_p95', lambda x: (x>0).sum()),
        total_anomaly_points=('anomaly_count_p95', 'sum')
    )
)

earliest_ts_comp = (
    trace_window[trace_window['anomaly_count_p95'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly_ts=('earliest_anomaly_ts_p95','min'))
)

trace_component_agg = (
    agg_basic
    .merge(distinct_trace_pair_count, on='cmdb_id', how='left')
    .merge(earliest_ts_comp, on='cmdb_id', how='left')
)

# ensure int types
trace_component_agg['num_trace_pairs_with_anomaly'] = trace_component_agg['num_trace_pairs_with_anomaly'].astype(int)
trace_component_agg['total_anomaly_points'] = trace_component_agg['total_anomaly_points'].astype(int)
trace_component_agg['distinct_trace_pair_count'] = trace_component_agg['distinct_trace_pair_count'].fillna(0).astype(int)

# top 20 components by total_anomaly_points
trace_component_agg_top20 = trace_component_agg.sort_values('total_anomaly_points', ascending=False).head(20)

# single integer: number of components with at least one trace P95 anomaly
num_trace_components_with_anomaly = int((trace_component_agg['num_trace_pairs_with_anomaly'] > 0).sum())

# For compactness, display shapes of full stored results plus the two top-20 tables and the integer
trace_thresholds.shape, trace_window.shape, trace_pair_top20, trace_component_agg_top20, num_trace_components_with_anomaly

```
Out[4]:
```
# Compose a concise plain-English summary using existing kernel variables
try:
    trace_thresholds
    trace_window
    trace_pair_top20
    trace_component_agg_top20
    num_trace_components_with_anomaly
except NameError:
    raise RuntimeError("Required trace variables are missing in the kernel.")

# Build summary lines
top_pair_examples = trace_pair_top20.head(5).reset_index(drop=True)
top_comp_examples = trace_component_agg_top20.head(5).reset_index(drop=True)

pair_lines = [
    f"{i+1}. {row['cmdb_id']} | {row['trace_name']} — anomaly_count_p95={row['anomaly_count_p95']}, points_in_window={row['points_in_window']}"
    for i, row in top_pair_examples.iterrows()
]
comp_lines = [
    f"{i+1}. {row['cmdb_id']} — total_anomaly_points={row['total_anomaly_points']}, num_trace_pairs_with_anomaly={row['num_trace_pairs_with_anomaly']}"
    for i, row in top_comp_examples.iterrows()
]

summary = (
    "Trace-level P95 anomaly summary (incident window 2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Total unique (cmdb_id, trace_name) pairs: {trace_thresholds.shape[0]}\n"
    f"- Components with at least one trace P95 anomaly: {num_trace_components_with_anomaly}\n"
    "- Top trace pairs by anomaly_count_p95 (examples):\n  " + "\n  ".join(pair_lines) + "\n"
    "- Top components by total anomaly points (examples):\n  " + "\n  ".join(comp_lines) + "\n"
    "- Observation: MG01 and MG02 show the highest component-level anomaly totals (each with 42 trace pairs with anomalies and 83 total anomaly points). "
    "Many trace pair anomalies are short but widespread within the 20:30–21:00 window; earliest anomalies occur around 20:32–20:37+08:00."
)

# Return compact outputs: summary string, small example tables, and the integer count
summary, top_pair_examples, top_comp_examples, num_trace_components_with_anomaly

The original code execution output of IPython Kernel is also provided below for reference:

((255, 5), (255, 8),       cmdb_id                       trace_name  total_count         p95         p90  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
59       MG01    trace.from_dockerB2.row_count           30  155.000000  154.100000                30                  3 2021-03-12 20:37:00+08:00
68       MG01      trace.to_dockerA1.row_count           30   97.000000   96.100000                30                  3 2021-03-12 20:46:00+08:00
95       MG02    trace.from_dockerA2.row_count           30   31.000000   29.200000                30                  3 2021-03-12 20:43:00+08:00
119      MG02      trace.to_dockerB2.row_count           30  118.000000  117.100000                30                  3 2021-03-12 20:34:00+08:00
191  dockerA1        trace.from_MG01.row_count           30   97.000000   96.100000                30                  3 2021-03-12 20:46:00+08:00
218  dockerA2          trace.to_MG02.row_count           30   31.000000   29.200000                30                  3 2021-03-12 20:43:00+08:00
242  dockerB2        trace.from_MG02.row_count           30  118.000000  117.100000                30                  3 2021-03-12 20:34:00+08:00
251  dockerB2          trace.to_MG01.row_count           30  155.000000  154.100000                30                  3 2021-03-12 20:37:00+08:00
3        IG01         trace.self.duration_mean           30    0.432312    0.394244                30                  2 2021-03-12 20:37:00+08:00
4        IG01          trace.self.duration_p95           30    2.157712    2.119975                30                  2 2021-03-12 20:37:00+08:00
5        IG01             trace.self.row_count           30  994.800000  958.600000                30                  2 2021-03-12 20:33:00+08:00
6        IG01  trace.to_Tomcat01.duration_mean           30    0.502841    0.396132                30                  2 2021-03-12 20:50:00+08:00
7        IG01   trace.to_Tomcat01.duration_p95           30    2.220210    2.115705                30                  2 2021-03-12 20:50:00+08:00
8        IG01      trace.to_Tomcat01.row_count           30  127.200000  125.000000                30                  2 2021-03-12 20:33:00+08:00
9        IG01  trace.to_Tomcat02.duration_mean           30    0.497532    0.428397                30                  2 2021-03-12 20:37:00+08:00
10       IG01   trace.to_Tomcat02.duration_p95           30    2.229565    2.073370                30                  2 2021-03-12 20:37:00+08:00
11       IG01      trace.to_Tomcat02.row_count           30  125.550000  125.000000                30                  2 2021-03-12 20:34:00+08:00
12       IG01  trace.to_Tomcat03.duration_mean           30    0.406987    0.363545                30                  2 2021-03-12 20:47:00+08:00
13       IG01   trace.to_Tomcat03.duration_p95           30    2.091285    2.056330                30                  2 2021-03-12 20:47:00+08:00
14       IG01      trace.to_Tomcat03.row_count           30  132.050000  118.800000                30                  2 2021-03-12 20:33:00+08:00,      cmdb_id  num_trace_pairs_with_anomaly  total_anomaly_points  distinct_trace_pair_count       earliest_anomaly_ts
3       MG02                            42                    83                         42 2021-03-12 20:32:00+08:00
2       MG01                            42                    83                         42 2021-03-12 20:33:00+08:00
11  dockerB2                            18                    35                         18 2021-03-12 20:32:00+08:00
7   Tomcat04                            18                    35                         18 2021-03-12 20:30:00+08:00
0       IG01                            18                    33                         18 2021-03-12 20:32:00+08:00
1       IG02                            18                    33                         18 2021-03-12 20:34:00+08:00
5   Tomcat02                            18                    33                         18 2021-03-12 20:30:00+08:00
4   Tomcat01                            18                    33                         18 2021-03-12 20:33:00+08:00
10  dockerB1                            18                    33                         18 2021-03-12 20:30:00+08:00
9   dockerA2                            15                    31                         15 2021-03-12 20:33:00+08:00
8   dockerA1                            15                    31                         15 2021-03-12 20:40:00+08:00
6   Tomcat03                            15                    30                         15 2021-03-12 20:33:00+08:00, 12)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse df_log if available; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure timezone-aware timestamp column 'ts' in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_log['ts'].dt.tz is None:
        df_log['ts'] = pd.to_datetime(df_log['ts']).dt.tz_localize(tz)
    else:
        df_log['ts'] = df_log['ts'].dt.tz_convert(tz)

# 1) Compute global thresholds for each (cmdb_id, log_name)
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        total_count=('value', 'count'),
        p95=('value', lambda x: x.quantile(0.95)),
        p90=('value', lambda x: x.quantile(0.90))
    )
)

# 2) Window filter: 2021-03-12 20:30 to 2021-03-12 21:00 UTC+8
start_ts = pd.Timestamp('2021-03-12 20:30:00', tz=tz)
end_ts = pd.Timestamp('2021-03-12 21:00:00', tz=tz)
df_log_win = df_log[(df_log['ts'] >= start_ts) & (df_log['ts'] <= end_ts)].copy()

# points_in_window per pair
points_in_window = (
    df_log_win
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(points_in_window=('value', 'count'))
)

# Merge global p95 into windowed rows and mark anomalies
df_log_win_p95 = df_log_win.merge(
    log_thresholds[['cmdb_id', 'log_name', 'p95']],
    on=['cmdb_id', 'log_name'],
    how='left'
)
df_log_win_p95['is_anom_p95'] = (df_log_win_p95['value'] >= df_log_win_p95['p95']).astype(int)

# Aggregate anomaly count and earliest anomaly timestamp per pair
anomaly_agg = (
    df_log_win_p95[df_log_win_p95['is_anom_p95'] == 1]
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        anomaly_count_p95=('is_anom_p95', 'sum'),
        earliest_anomaly_ts_p95=('ts', 'min')
    )
)

# 2) Build log_window: left-join thresholds with window aggregates so every pair appears
log_window = (
    log_thresholds
    .merge(points_in_window, on=['cmdb_id', 'log_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'log_name'], how='left')
)

# fill missing window values appropriately
log_window['points_in_window'] = log_window['points_in_window'].fillna(0).astype(int)
log_window['anomaly_count_p95'] = log_window['anomaly_count_p95'].fillna(0).astype(int)
# earliest_anomaly_ts_p95 remains NaT where no anomaly

# 3a) log_pair_top20: top 20 pairs sorted by anomaly_count_p95 desc (tie-breaker: points_in_window)
log_pair_top20 = (
    log_window
    .sort_values(['anomaly_count_p95', 'points_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','log_name','total_count','p95','p90','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']]
    .head(20)
)

# 3b) component-level aggregation
distinct_log_pair_count = (
    log_thresholds
    .groupby('cmdb_id', as_index=False)
    .agg(distinct_log_pair_count=('log_name', 'count'))
)

agg_basic = (
    log_window
    .groupby('cmdb_id', as_index=False)
    .agg(
        num_log_pairs_with_anomaly=('anomaly_count_p95', lambda x: (x>0).sum()),
        total_anomaly_points=('anomaly_count_p95', 'sum')
    )
)

earliest_ts_comp = (
    log_window[log_window['anomaly_count_p95'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly_ts=('earliest_anomaly_ts_p95', 'min'))
)

log_component_agg = (
    agg_basic
    .merge(distinct_log_pair_count, on='cmdb_id', how='left')
    .merge(earliest_ts_comp, on='cmdb_id', how='left')
)

# ensure int types
log_component_agg['num_log_pairs_with_anomaly'] = log_component_agg['num_log_pairs_with_anomaly'].astype(int)
log_component_agg['total_anomaly_points'] = log_component_agg['total_anomaly_points'].astype(int)
log_component_agg['distinct_log_pair_count'] = log_component_agg['distinct_log_pair_count'].fillna(0).astype(int)

# top 20 components by total_anomaly_points
log_component_agg_top20 = log_component_agg.sort_values('total_anomaly_points', ascending=False).head(20)

# single integer: number of components with at least one log P95 anomaly
num_log_components_with_anomaly = int((log_component_agg['num_log_pairs_with_anomaly'] > 0).sum())

# Display compact outputs: shapes and the two top-20 tables and the integer
log_thresholds.shape, log_window.shape, log_pair_top20, log_component_agg_top20, num_log_components_with_anomaly

```
Out[5]:
```
Summary of log-level P95 anomaly analysis (incident window 2021-03-12 20:30→21:00 UTC+8):

- Data size:
  - Unique (cmdb_id, log_name) pairs analyzed: 16
  - Pairs present in the incident window: 16

- Key findings (top pair examples):
  - Tomcat01 | log.error_count — total_count=31, p95=0.0, points_in_window=31, anomaly_count_p95=31, earliest=2021-03-12 20:30:00+08:00
  - Tomcat02 | log.error_count — same pattern: anomaly_count_p95=31, earliest=20:30+08:00
  - Tomcat03 | log.error_count — anomaly_count_p95=31, earliest=20:30+08:00
  - Tomcat04, apache01, apache02 | log.error_count — all similarly show anomaly_count_p95=31 with earliest at 20:30+08:00
  - log.row_count pairs (Tomcat and apache) show smaller anomaly_count_p95 (typically 2) with earliest anomalies around 20:33–20:34+08:00
  - log.gc_pause_max shows a few isolated anomalies (1–2 points) with earliest times around 20:36–20:53+08:00

- Component-level summary (top components by total anomaly points):
  - Tomcat03 — num_log_pairs_with_anomaly=3, total_anomaly_points=35, distinct_log_pair_count=3, earliest=2021-03-12 20:30:00+08:00
  - Tomcat01 — num=3, total_anomaly_points=34, earliest=20:30+08:00
  - Tomcat02 — num=3, total_anomaly_points=34, earliest=20:30+08:00
  - Tomcat04 — num=3, total_anomaly_points=34, earliest=20:30+08:00
  - apache01 — num=2, total_anomaly_points=33, earliest=20:30+08:00
  - apache02 — num=2, total_anomaly_points=33, earliest=20:30+08:00

- Count of affected components:
  - Number of components with at least one log P95 anomaly in the window: 6

- Notes / interpretation:
  - Many log.error_count entries have p95=0 and therefore register as anomalies across all 31 window points (anomaly_count_p95=31). This makes those "anomalies" less informative because the threshold is zero; treat those with caution.
  - More meaningful anomalies (non-zero p95) appear in log.row_count and log.gc_pause_max where isolated spikes occurred (earliest ~20:33–20:53+08:00).
  - Earliest widespread indications appear at 20:30:00+08:00 for error_count metrics across Tomcat/apache.

The original code execution output of IPython Kernel is also provided below for reference:

((16, 5), (16, 8),      cmdb_id          log_name  total_count       p95      p90  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
0   Tomcat01   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
3   Tomcat02   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
6   Tomcat03   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
9   Tomcat04   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
12  apache01   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
14  apache02   log.error_count           31    0.0000    0.000                31                 31 2021-03-12 20:30:00+08:00
2   Tomcat01     log.row_count           31  248.5000  243.000                31                  2 2021-03-12 20:33:00+08:00
5   Tomcat02     log.row_count           31  243.5000  238.000                31                  2 2021-03-12 20:34:00+08:00
8   Tomcat03     log.row_count           31  258.5000  232.000                31                  2 2021-03-12 20:34:00+08:00
11  Tomcat04     log.row_count           31  244.5000  238.000                31                  2 2021-03-12 20:34:00+08:00
13  apache01     log.row_count           31  494.5000  461.000                31                  2 2021-03-12 20:34:00+08:00
15  apache02     log.row_count           31  484.0000  467.000                31                  2 2021-03-12 20:33:00+08:00
7   Tomcat03  log.gc_pause_max           11    0.0900    0.090                11                  2 2021-03-12 20:53:00+08:00
4   Tomcat02  log.gc_pause_max           20    0.1305    0.103                20                  1 2021-03-12 20:36:00+08:00
1   Tomcat01  log.gc_pause_max           11    0.0850    0.070                11                  1 2021-03-12 20:51:00+08:00
10  Tomcat04  log.gc_pause_max           11    0.1200    0.110                11                  1 2021-03-12 20:36:00+08:00,     cmdb_id  num_log_pairs_with_anomaly  total_anomaly_points  distinct_log_pair_count       earliest_anomaly_ts
2  Tomcat03                           3                    35                        3 2021-03-12 20:30:00+08:00
0  Tomcat01                           3                    34                        3 2021-03-12 20:30:00+08:00
1  Tomcat02                           3                    34                        3 2021-03-12 20:30:00+08:00
3  Tomcat04                           3                    34                        3 2021-03-12 20:30:00+08:00
4  apache01                           2                    33                        2 2021-03-12 20:30:00+08:00
5  apache02                           2                    33                        2 2021-03-12 20:30:00+08:00, 6)```
```

In [ ]:
# In[6]:

import re
import pandas as pd

# Reuse trace_window from kernel
try:
    trace_window
except NameError:
    raise RuntimeError("trace_window not found in kernel. Run trace analysis step first.")

# 1) Filter trace pairs for calls to candidate services and with anomaly_count_p95 > 0
pattern = r'to_(Tomcat|Mysql|Redis|apache)[0-9]*'  # match target prefixes with optional digits
mask = trace_window['trace_name'].str.contains(pattern, case=False, regex=True, na=False) & (trace_window['anomaly_count_p95'] > 0)
trace_call_anomalies = trace_window[mask].copy()

# 2) Extract callee service name (e.g., Tomcat01, Mysql01, Redis02, apache01) using a broader regex
# capture the full token after 'to_' up to the next non-alphanumeric/underscore/dot character
trace_call_anomalies['callee'] = trace_call_anomalies['trace_name'].str.extract(r'to_([A-Za-z]+[0-9]*)', expand=False)

# 3) Prepare compact output table sorted by earliest anomaly timestamp ascending, limited to 50 rows
trace_call_top50 = (
    trace_call_anomalies
    .sort_values('earliest_anomaly_ts_p95', ascending=True)
    .loc[:, ['cmdb_id','trace_name','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']]
    .head(50)
)

# 4) Count distinct callee services referenced
num_distinct_callees = int(trace_call_anomalies['callee'].dropna().nunique())

# Return compact results
trace_call_top50, num_distinct_callees

```
Out[6]:
```
# Compose a concise plain-English summary string using existing kernel variables
try:
    trace_call_top50
    trace_call_anomalies
    num_distinct_callees
except NameError:
    raise RuntimeError("Required variables (trace_call_top50, trace_call_anomalies, num_distinct_callees) are missing in the kernel.")

total_pairs = int(trace_call_anomalies.shape[0])
callees = sorted(trace_call_anomalies['callee'].dropna().unique().tolist())
earliest_ts = trace_call_anomalies['earliest_anomaly_ts_p95'].min()
# prepare up to 5 example lines
examples = []
for _, r in trace_call_top50.head(5).iterrows():
    examples.append(f"{r['cmdb_id']} | {r['trace_name']} — anomalies={int(r['anomaly_count_p95'])}, earliest={r['earliest_anomaly_ts_p95']}")

summary = (
    f"Trace-call anomalies to candidate services (window 2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Total trace pairs matching 'to_Tomcat/to_Mysql/to_Redis/to_apache' with P95 anomalies: {total_pairs}\n"
    f"- Number of distinct callee services referenced: {num_distinct_callees} ({', '.join(callees)})\n"
    f"- Earliest anomaly observed at: {earliest_ts}\n"
    f"- Examples (up to 5):\n  " + "\n  ".join(examples) + "\n\n"
    f"Interpretation: The anomalies are predominantly calls to Tomcat services (multiple trace pairs per Tomcat instance), "
    f"with earliest occurrences starting around 20:32+08:00 and additional anomalies throughout the 20:30–21:00 window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
17    IG01      trace.to_Tomcat04.row_count                30                  2 2021-03-12 20:32:00+08:00
8     IG01      trace.to_Tomcat01.row_count                30                  2 2021-03-12 20:33:00+08:00
14    IG01      trace.to_Tomcat03.row_count                30                  2 2021-03-12 20:33:00+08:00
32    IG02      trace.to_Tomcat03.row_count                30                  2 2021-03-12 20:34:00+08:00
11    IG01      trace.to_Tomcat02.row_count                30                  2 2021-03-12 20:34:00+08:00
..     ...                              ...               ...                ...                       ...
15    IG01  trace.to_Tomcat04.duration_mean                30                  2 2021-03-12 20:48:00+08:00
24    IG02  trace.to_Tomcat01.duration_mean                30                  2 2021-03-12 20:48:00+08:00
33    IG02  trace.to_Tomcat04.duration_mean                30                  2 2021-03-12 20:49:00+08:00
7     IG01   trace.to_Tomcat01.duration_p95                30                  2 2021-03-12 20:50:00+08:00
6     IG01  trace.to_Tomcat01.duration_mean                30                  2 2021-03-12 20:50:00+08:00

[24 rows x 5 columns], 4)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd

# Reuse trace_window from kernel
try:
    trace_window
except NameError:
    raise RuntimeError("trace_window not found in kernel. Run trace analysis step first.")

# Define sets
tomcat_ids = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']
db_ids = ['Mysql01','Mysql02','Redis01','Redis02']

# Build masks for conditions
mask_a = trace_window['cmdb_id'].isin(tomcat_ids) & trace_window['trace_name'].str.contains(r'trace\.self|trace\.from_|trace\.to_', case=False, regex=True, na=False)
mask_b = trace_window['trace_name'].str.contains(r'to_Mysql|to_Redis', case=False, regex=True, na=False)
mask_c = trace_window['cmdb_id'].isin(db_ids) & trace_window['trace_name'].str.contains(r'trace\.self|trace\.from_|trace\.to_', case=False, regex=True, na=False)

# Combine and filter anomaly_count_p95 > 0
mask = (mask_a | mask_b | mask_c) & (trace_window['anomaly_count_p95'] > 0)
trace_tomcat_db_window = trace_window[mask].copy()

# Select required columns and sort by earliest_anomaly_ts_p95 ascending, limit to top 30
trace_tomcat_db_window = trace_tomcat_db_window.loc[:, ['cmdb_id','trace_name','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']]
trace_tomcat_db_window = trace_tomcat_db_window.sort_values('earliest_anomaly_ts_p95', ascending=True).head(30)

# Compute distinct counts
num_tomcat_instances_with_anomaly = int(trace_tomcat_db_window[trace_tomcat_db_window['cmdb_id'].isin(tomcat_ids)]['cmdb_id'].nunique())
num_dbcache_instances_with_anomaly = int(trace_tomcat_db_window[trace_tomcat_db_window['cmdb_id'].isin(db_ids)]['cmdb_id'].nunique())

# Return compact outputs (DataFrame head limited to 30 already)
trace_tomcat_db_window, num_tomcat_instances_with_anomaly, num_dbcache_instances_with_anomaly

```
Out[7]:
```
# Summarize selected trace-window anomalies for Tomcat and DB/cache related traces
try:
    trace_tomcat_db_window
    num_tomcat_instances_with_anomaly
    num_dbcache_instances_with_anomaly
except NameError:
    raise RuntimeError("Required kernel variables are missing. Ensure trace_tomcat_db_window and the two integer counts exist.")

total_selected = int(trace_tomcat_db_window.shape[0])
earliest_ts = trace_tomcat_db_window['earliest_anomaly_ts_p95'].min()
top_trace_examples = trace_tomcat_db_window['trace_name'].value_counts().head(5).index.tolist()
tomcat_list = sorted(trace_tomcat_db_window[trace_tomcat_db_window['cmdb_id'].str.startswith('Tomcat')]['cmdb_id'].unique().tolist())

summary = (
    "Selected trace-window anomaly summary (2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Total matching trace entries with P95 anomalies: {total_selected}\n"
    f"- Distinct Tomcat instances with any trace-window anomaly: {num_tomcat_instances_with_anomaly} ({', '.join(tomcat_list)})\n"
    f"- Distinct DB/cache (Mysql/Redis) instances with any trace-window anomaly in this selection: {num_dbcache_instances_with_anomaly}\n"
    f"- Earliest anomaly timestamp in this selection: {earliest_ts}\n"
    f"- Most common trace names in this selection (examples): {', '.join(top_trace_examples)}\n\n"
    "Interpretation: All four Tomcat instances show trace-level anomalies (examples include from_root, from_IG01, to_MG02) starting around 20:30+08:00. "
    "No direct Mysql/Redis trace anomalies were present in this filtered set."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                     trace_name  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
146  Tomcat02      trace.from_root.row_count                 2                  1 2021-03-12 20:30:00+08:00
178  Tomcat04   trace.from_root.duration_p95                 2                  2 2021-03-12 20:30:00+08:00
177  Tomcat04  trace.from_root.duration_mean                 2                  2 2021-03-12 20:30:00+08:00
188  Tomcat04        trace.to_MG02.row_count                30                  2 2021-03-12 20:32:00+08:00
173  Tomcat04      trace.from_IG01.row_count                30                  2 2021-03-12 20:32:00+08:00
..        ...                            ...               ...                ...                       ...
145  Tomcat02   trace.from_root.duration_p95                 2                  1 2021-03-12 20:35:00+08:00
187  Tomcat04     trace.to_MG02.duration_p95                30                  2 2021-03-12 20:37:00+08:00
153  Tomcat02    trace.to_MG02.duration_mean                30                  2 2021-03-12 20:37:00+08:00
151  Tomcat02     trace.to_MG01.duration_p95                30                  2 2021-03-12 20:37:00+08:00
154  Tomcat02     trace.to_MG02.duration_p95                30                  2 2021-03-12 20:37:00+08:00

[30 rows x 5 columns], 4, 0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing metrics_window from kernel
try:
    metrics_window
except NameError:
    raise RuntimeError("metrics_window not found in kernel. Please run the metric analysis steps first.")

# Ensure timezone variable (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Define Tomcat instances
tomcat_ids = ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04']

# Filter metrics_window for Tomcat instances (keep full rows in memory, display compact)
tomcat_metric_anomalies_full = metrics_window[metrics_window['cmdb_id'].isin(tomcat_ids)].copy()

# Select and order requested columns
cols = ['cmdb_id','kpi_name','total_count','p95','p90','points_in_window','anomaly_count_p95','earliest_anomaly_ts_p95']
# Some rows might not have all columns if upstream differed; ensure they exist
for c in cols:
    if c not in tomcat_metric_anomalies_full.columns:
        tomcat_metric_anomalies_full[c] = pd.NA

# Sort by anomaly_count_p95 descending, then points_in_window descending as tiebreaker
tomcat_metric_anomalies_full = tomcat_metric_anomalies_full.sort_values(['anomaly_count_p95','points_in_window'], ascending=[False, False])

# Limit to top 50 for compact display; store full table in kernel variable name as requested
tomcat_metric_anomalies = tomcat_metric_anomalies_full.loc[:, cols].head(50)

# 1) number of distinct Tomcat KPI pairs that had anomaly_count_p95 > 0
num_distinct_tomcat_kpi_pairs_with_anomaly = int((tomcat_metric_anomalies_full['anomaly_count_p95'] > 0).sum())

# 2) earliest earliest_anomaly_ts_p95 among all Tomcat KPI pairs (consider only those with anomaly_count_p95>0)
if (tomcat_metric_anomalies_full['anomaly_count_p95'] > 0).any():
    earliest_t = tomcat_metric_anomalies_full.loc[tomcat_metric_anomalies_full['anomaly_count_p95'] > 0, 'earliest_anomaly_ts_p95'].min()
    # Ensure timezone awareness (if it's a string or naive, try to coerce)
    try:
        if pd.isna(earliest_t):
            earliest_tomcat_metric_anomaly_ts = pd.NaT
        else:
            earliest_tomcat_metric_anomaly_ts = pd.to_datetime(earliest_t).tz_convert(tz) if getattr(earliest_t, 'tzinfo', None) else pd.to_datetime(earliest_t).tz_localize(tz)
    except Exception:
        # fallback: keep as-is
        earliest_tomcat_metric_anomaly_ts = earliest_t
else:
    earliest_tomcat_metric_anomaly_ts = pd.NaT

# Return compact outputs: the limited dataframe and the two integers/timestamp
tomcat_metric_anomalies, num_distinct_tomcat_kpi_pairs_with_anomaly, earliest_tomcat_metric_anomaly_ts

```
Out[8]:
```
# Compose a concise plain-English summary of the Tomcat metric anomalies using existing kernel variables
try:
    tomcat_metric_anomalies
    num_distinct_tomcat_kpi_pairs_with_anomaly
    earliest_tomcat_metric_anomaly_ts
except NameError:
    raise RuntimeError("Required kernel variables are missing: tomcat_metric_anomalies, num_distinct_tomcat_kpi_pairs_with_anomaly, earliest_tomcat_metric_anomaly_ts")

# Build example lines from the top of the tomcat_metric_anomalies table
examples = []
for i, row in tomcat_metric_anomalies.head(5).iterrows():
    examples.append(f"{row['cmdb_id']} | {row['kpi_name'][:80]}... — anomalies={int(row['anomaly_count_p95'])}, earliest={row['earliest_anomaly_ts_p95']}")

summary = (
    "Tomcat metrics anomaly summary (incident window 2021-03-12 20:30→21:00 UTC+8):\n"
    f"- Distinct Tomcat (cmdb_id,kpi_name) pairs with P95 anomalies: {num_distinct_tomcat_kpi_pairs_with_anomaly}\n"
    f"- Earliest Tomcat metric anomaly observed at: {earliest_tomcat_metric_anomaly_ts}\n"
    f"- Note: Many Tomcat KPI pairs show anomalies across the entire window (points_in_window=31 and anomaly_count_p95=31),\n"
    "  indicating widespread metric elevations reaching or exceeding their global P95 thresholds starting at 20:30+08:00.\n"
    "- Examples (top 5):\n  " + "\n  ".join(examples) + "\n\n"
    "Interpretation: All Tomcat instances exhibit numerous metric KPIs hitting P95 during the window; investigate Tomcat request/error/session KPIs and upstream/downstream traces for root cause."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  total_count       p95       p90  points_in_window  anomaly_count_p95   earliest_anomaly_ts_p95
1172  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...           31     987.0     987.0                31                 31 2021-03-12 20:30:00+08:00
1173  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...           31  405396.0  405396.0                31                 31 2021-03-12 20:30:00+08:00
1176  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
1177  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
1178  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
...        ...                                                ...          ...       ...       ...               ...                ...                       ...
1500  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...           31  407896.0  407896.0                31                 31 2021-03-12 20:30:00+08:00
1503  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
1504  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
1505  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00
1506  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter           31       0.0       0.0                31                 31 2021-03-12 20:30:00+08:00

[50 rows x 8 columns], 437, Timestamp('2021-03-12 20:30:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware ts column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_metric['ts'].dt.tz is None:
        df_metric['ts'] = pd.to_datetime(df_metric['ts']).dt.tz_localize(tz)
    else:
        df_metric['ts'] = pd.to_datetime(df_metric['ts']).dt.tz_convert(tz)

# Incident window
start_ts = pd.Timestamp('2021-03-12 20:30:00', tz=tz)
end_ts = pd.Timestamp('2021-03-12 21:00:00', tz=tz)

# Tomcat ids
tomcat_ids = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# 1) Filter metric.csv to Tomcat ids and incident window
df_tomcat_win = df_metric[(df_metric['cmdb_id'].isin(tomcat_ids)) & (df_metric['ts'] >= start_ts) & (df_metric['ts'] <= end_ts)].copy()

# 2) Aggregate stats in window per (cmdb_id, kpi_name)
agg_window = df_tomcat_win.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    points_in_window=('value','count'),
    max_value_in_window=('value','max'),
    mean_value_in_window=('value','mean')
)

# Obtain global P95 and related fields from previously computed tomcat candidate set.
# Prefer tomcat_metric_anomalies_full if available, else tomcat_metric_anomalies, else metrics_window filtered.
p95_source = None
try:
    tomcat_metric_candidates = tomcat_metric_anomalies_full.loc[:, ['cmdb_id','kpi_name','p95','anomaly_count_p95','earliest_anomaly_ts_p95']]
    p95_source = 'tomcat_metric_anomalies_full'
except NameError:
    try:
        tomcat_metric_candidates = tomcat_metric_anomalies.loc[:, ['cmdb_id','kpi_name','p95','anomaly_count_p95','earliest_anomaly_ts_p95']]
        p95_source = 'tomcat_metric_anomalies'
    except NameError:
        # fallback to metrics_window filtered to Tomcats
        tomcat_metric_candidates = metrics_window[metrics_window['cmdb_id'].isin(tomcat_ids)].loc[:, ['cmdb_id','kpi_name','p95','anomaly_count_p95','earliest_anomaly_ts_p95']]
        p95_source = 'metrics_window_filtered'

# 2b) Merge global P95 and anomaly_count/earliest from the candidate set into the window aggregates.
tomcat_kpi_severity = (
    agg_window
    .merge(tomcat_metric_candidates, on=['cmdb_id','kpi_name'], how='left')
)

# Ensure numeric columns
tomcat_kpi_severity['p95'] = pd.to_numeric(tomcat_kpi_severity['p95'], errors='coerce').fillna(0.0)
tomcat_kpi_severity['anomaly_count_p95'] = pd.to_numeric(tomcat_kpi_severity['anomaly_count_p95'], errors='coerce').fillna(0).astype(int)

# Compute percent_breach
def compute_percent_breach(row):
    p95 = row['p95']
    maxv = row['max_value_in_window']
    if pd.isna(p95):
        return np.nan
    if p95 > 0:
        return (maxv - p95) / p95
    # p95 == 0
    if p95 == 0:
        if maxv > 0:
            return np.inf
        else:
            return 0.0
    return np.nan

tomcat_kpi_severity['percent_breach'] = tomcat_kpi_severity.apply(compute_percent_breach, axis=1)

# Keep earliest_anomaly_ts_p95 as-is (may be NaT)
# 3) Identify high-severity KPI pairs:
# condition A: percent_breach >= 0.5
cond_a = tomcat_kpi_severity['percent_breach'] >= 0.5

# condition B: anomaly_count_p95 == points_in_window == 31 and p95 == 0 and max_value_in_window > 0
cond_b = (
    (tomcat_kpi_severity['anomaly_count_p95'] == tomcat_kpi_severity['points_in_window']) &
    (tomcat_kpi_severity['points_in_window'] == 31) &
    (tomcat_kpi_severity['p95'] == 0) &
    (tomcat_kpi_severity['max_value_in_window'] > 0)
)

tomcat_kpi_severity['high_severity'] = cond_a | cond_b

# 4) Prepare top-20 table sorted by percent_breach desc (treat inf as highest)
# Replace NaN percent_breach with -inf for sorting lower
sort_series = tomcat_kpi_severity['percent_breach'].replace({np.nan: -np.inf})
tomcat_kpi_severity_top20 = (
    tomcat_kpi_severity
    .assign(_sort_pb=sort_series)
    .sort_values(['_sort_pb','anomaly_count_p95'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','points_in_window','max_value_in_window','mean_value_in_window','p95','percent_breach','anomaly_count_p95','earliest_anomaly_ts_p95','high_severity']]
    .head(20)
    .drop(columns=['high_severity'])  # not requested in display columns, keep full table stored in tomcat_kpi_severity
)

# Count high severity KPI pairs
count_high_severity = int(tomcat_kpi_severity['high_severity'].sum())

# Store full severity table in kernel variable name as requested (tomcat_kpi_severity already set)
# Return compact outputs (top20 and the integer)
tomcat_kpi_severity_top20, count_high_severity

```
Out[9]:
```
Summary of Tomcat KPI severity (incident window 2021-03-12 20:30→21:00 UTC+8):

- High-severity KPI count:
  - Number of Tomcat KPI pairs meeting the high-severity criteria: 10

- Top high-severity examples (from the severity ranking, percent_breach descending):
  1) Tomcat04 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — percent_breach ≈ 185.7% (max=1.0 vs p95=0.35), earliest anomaly 2021-03-12 21:00:00+08:00  
  2) Tomcat03 | OSLinux-CPU_CPU_CPUWio — percent_breach ≈ 105.4%, earliest 2021-03-12 20:49:00+08:00  
  3) Tomcat04 | OSLinux-CPU_CPU_CPUWio — percent_breach ≈ 85.6%, earliest 2021-03-12 20:52:00+08:00  
  4) Tomcat01 | local disk read/write and I/O KPIs (multiple DSK* metrics) — percent_breach ~81.8%, earliest ~20:31–20:38+08:00  
  5) Tomcat01 | Tomcat-Threads...CurrentThread (thread count) — percent_breach ≈ 66.7%, anomaly_count_p95=5, earliest 20:33+08:00  
  - Additional entries include other local-disk and CPU single-core metrics with percent_breach values from ~47% down to ~29%, and various earliest timestamps between ~20:31 and 21:00+08:00.

- Earliest Tomcat metric anomaly observed: 2021-03-12 20:31:00+08:00 (some disk / DSK metrics and request/thread KPIs show anomalies at ~20:31–20:33).

- Interpretation / actionable notes:
  - The highest severity signals are centered on I/O and CPU-related KPIs (local disk I/O, CPU wait, per-core CPU utilization) and also Tomcat thread/request KPIs. Several Tomcat KPIs show sustained breaches across the whole window (points_in_window=31 with many anomalies), and others show strong spikes (percent_breach >= 50%).
  - Tomcat thread/request KPIs and disk I/O spikes coincide early in the window (~20:31–20:33), while some network/close-wait and CPU-wait anomalies appear later (up to ~21:00).
  - Recommended next steps: prioritize investigation of Tomcat request/thread exhaustion and host-level disk I/O and CPU wait on the affected Tomcat nodes; correlate with trace-level anomalies and logs to find upstream callers or resource contention.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  points_in_window  max_value_in_window  mean_value_in_window        p95  percent_breach  anomaly_count_p95   earliest_anomaly_ts_p95
390  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                14               1.0000              0.071429   0.350000        1.857143                  1 2021-03-12 21:00:00+08:00
229  Tomcat03                             OSLinux-CPU_CPU_CPUWio                17               0.5448              0.094176   0.265200        1.054299                  1 2021-03-12 20:49:00+08:00
338  Tomcat04                             OSLinux-CPU_CPU_CPUWio                17               0.2453              0.050859   0.132180        0.855803                  1 2021-03-12 20:52:00+08:00
44   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                10               0.6667              0.066670   0.366685        0.818182                  1 2021-03-12 20:38:00+08:00
260  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                10               0.1833              0.018330   0.100815        0.818182                  1 2021-03-12 20:31:00+08:00
261  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                10               5.8667              0.586670   3.226685        0.818182                  1 2021-03-12 20:31:00+08:00
41   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                10               5.0000              0.500000   2.750000        0.818182                  1 2021-03-12 20:36:00+08:00
43   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                10               0.0667              0.006670   0.036685        0.818182                  1 2021-03-12 20:38:00+08:00
55   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps                10               1.0000              0.100000   0.550000        0.818182                  1 2021-03-12 20:53:00+08:00
105  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                31               5.0000              1.032258   3.000000        0.666667                  5 2021-03-12 20:33:00+08:00
157  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                 9               5.0000              0.666667   3.400000        0.470588                  1 2021-03-12 20:47:00+08:00
367  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                 9               5.0000              0.666667   3.400000        0.470588                  1 2021-03-12 20:51:00+08:00
267  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps                10               6.0000              1.000000   4.200000        0.428571                  1 2021-03-12 20:31:00+08:00
152  Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                10              45.0000             15.700000  32.850000        0.369863                  1 2021-03-12 20:39:00+08:00
258  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                 9               6.0000              0.888889   4.400000        0.363636                  1 2021-03-12 20:59:00+08:00
268  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                 9               0.8000              0.155556   0.600000        0.333333                  1 2021-03-12 20:33:00+08:00
324  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                31               4.0000              0.935484   3.000000        0.333333                  4 2021-03-12 20:33:00+08:00
6    Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil                10               5.9244              2.349850   4.471620        0.324889                  1 2021-03-12 20:46:00+08:00
184  Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                20               0.4000              0.305000   0.305000        0.311475                  1 2021-03-12 20:42:00+08:00
219  Tomcat03                    OSLinux-CPU_CPU-1_SingleCpuUtil                 9             100.0000             21.784167  77.246440        0.294558                  1 2021-03-12 20:59:00+08:00, 10)```
```